# Databricks Smart Mobility Lakehouse: Medallion Architecture Demo

**Author:** Tran Chung (UTC Master 2 ISC AOS / Ingénieur Génie Informatique)

---

### Overview
This notebook demonstrates a production-grade **Lakehouse Architecture** for urban smart mobility events (e-bikes, e-scooters, electric car sharing across Paris, Lyon, Lille, Toulouse, Nantes):
1. **Bronze Layer:** Raw ingestion of semi-structured events with audit metadata (`_ingest_timestamp`, `_source_file`, `_batch_id`).
2. **Silver Layer:** Cleansing, deduplication, $\text{CO}_2$ carbon offset feature engineering, station metadata enrichment, and **ACID `MERGE INTO` (CDC Upsert)**.
3. **Gold Layer:** High-value analytics aggregated for city mobility dashboards, station turnover rates, and **Z-Order Multi-dimensional indexing**.
4. **Delta Lake Governance:** Transaction Log inspection (`DESCRIBE HISTORY`) and Time Travel queries.

## 1. Setup & Environment Configuration

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    avg,
    col,
    count,
    countDistinct,
    current_timestamp,
    input_file_name,
    lit,
    to_date,
    to_timestamp,
    unix_timestamp,
    when,
)
from pyspark.sql.functions import round as spark_round
from pyspark.sql.functions import sum as spark_sum

# When running inside Databricks, the SparkSession is already active via `spark`.
# If running locally:
spark = (
    SparkSession.builder.appName("DatabricksMobilityLakehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

print(f"Spark Version: {spark.version}")

## 2. Bronze Layer: Raw Ingestion with Audit Columns

Preserve raw event fidelity without destructive transforms while appending lineage tracking columns.

In [ ]:
# Define Lakehouse storage paths (or Unity Catalog volume paths)
bronze_trips_path = "data/lakehouse/bronze/trips"
bronze_stations_path = "data/lakehouse/bronze/stations"

# Ingest reference stations
stations_df = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv("data/raw/stations.csv")
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_source_file", input_file_name())
)

stations_df.write.format("delta").mode("overwrite").save(bronze_stations_path)
print(f"Stations Bronze Table created with {stations_df.count()} rows.")
display(stations_df.limit(5))

In [ ]:
# Ingest Trips Batch 1
raw_trips_b1 = (
    spark.read.option("multiline", "true")
    .json("data/raw/trips_batch_20260101.json")
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_source_file", input_file_name())
    .withColumn("_batch_id", lit("batch_20260101"))
)

raw_trips_b1.write.format("delta").mode("append").option("mergeSchema", "true").save(
    bronze_trips_path
)
print(f"Ingested {raw_trips_b1.count()} trips into Bronze.")

## 3. Silver Layer: Cleansing, Feature Engineering & CDC Upsert

We calculate trip durations, compute carbon savings ($\text{CO}_2$ offset vs combustion vehicles), filter anomalies, and perform an ACID `MERGE INTO`.

In [ ]:
def clean_and_enrich_bronze(df):
    # Cast timestamps and compute duration
    ts_df = (
        df.withColumn("start_ts", to_timestamp(col("start_time")))
        .withColumn("end_ts", to_timestamp(col("end_time")))
        .withColumn("trip_date", to_date(col("start_ts")))
        .withColumn(
            "duration_min",
            spark_round(
                (unix_timestamp("end_ts") - unix_timestamp("start_ts")) / 60.0, 2
            ),
        )
    )

    # Quality gates: positive distance, positive duration (< 12h)
    valid_df = ts_df.filter(
        (col("distance_km") > 0.0)
        & (col("duration_min") > 0.0)
        & (col("duration_min") < 720.0)
    )

    # Deduplicate on business primary key
    dedup_df = valid_df.dropDuplicates(["trip_id"])

    # Feature engineering: CO2 offset (kg)
    enriched_df = dedup_df.withColumn(
        "co2_saved_kg",
        spark_round(
            when(col("vehicle_type") == "ebike", col("distance_km") * 0.130)
            .when(col("vehicle_type") == "escooter", col("distance_km") * 0.125)
            .when(col("vehicle_type") == "ev_car", col("distance_km") * 0.090)
            .otherwise(col("distance_km") * 0.100),
            3,
        ),
    )

    # Join stations to resolve start/end city & station name
    st_ref = spark.read.format("delta").load(bronze_stations_path)
    return (
        enriched_df.join(
            st_ref.select(
                col("station_id").alias("s_id"),
                col("city").alias("start_city"),
                col("station_name").alias("start_station_name"),
            ),
            col("start_station") == col("s_id"),
            "left",
        )
        .drop("s_id")
        .join(
            st_ref.select(
                col("station_id").alias("e_id"),
                col("city").alias("end_city"),
                col("station_name").alias("end_station_name"),
            ),
            col("end_station") == col("e_id"),
            "left",
        )
        .drop("e_id")
    )


silver_trips_path = "data/lakehouse/silver/trips_enriched"
silver_b1 = clean_and_enrich_bronze(raw_trips_b1)
silver_b1.write.format("delta").mode("overwrite").partitionBy("trip_date").save(
    silver_trips_path
)
print("Silver table initialized.")

### Incremental Change Data Capture (CDC) with Delta Lake `MERGE INTO`

Now we ingest Batch 2 which contains new trips, duplicates, and an update for `TRP-1010` (fare update & completed status).

In [ ]:
raw_trips_b2 = (
    spark.read.option("multiline", "true")
    .json("data/raw/trips_batch_20260102.json")
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_source_file", input_file_name())
    .withColumn("_batch_id", lit("batch_20260102"))
)

raw_trips_b2.write.format("delta").mode("append").option("mergeSchema", "true").save(
    bronze_trips_path
)
silver_b2 = clean_and_enrich_bronze(raw_trips_b2)

# Delta Lake ACID MERGE INTO (Upsert)
target_table = DeltaTable.forPath(spark, silver_trips_path)
(
    target_table.alias("target")
    .merge(silver_b2.alias("source"), "target.trip_id = source.trip_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print("Successfully performed Delta Lake ACID MERGE INTO.")

## 4. Gold Layer: Executive Business KPIs & Z-Order Optimization

In [ ]:
silver_final = spark.read.format("delta").load(silver_trips_path)

gold_kpis = (
    silver_final.groupBy("trip_date", "start_city", "vehicle_type")
    .agg(
        count("trip_id").alias("total_trips"),
        countDistinct("user_id").alias("unique_riders"),
        spark_round(spark_sum("distance_km"), 2).alias("total_distance_km"),
        spark_round(avg("duration_min"), 2).alias("avg_duration_min"),
        spark_round(spark_sum("fare_eur"), 2).alias("total_revenue_eur"),
        spark_round(spark_sum("co2_saved_kg"), 3).alias("total_co2_saved_kg"),
    )
    .withColumnRenamed("start_city", "city")
)

gold_kpis_path = "data/lakehouse/gold/daily_city_kpis"
gold_kpis.write.format("delta").mode("overwrite").partitionBy("trip_date").save(
    gold_kpis_path
)

# Databricks Z-Order Optimization
# spark.sql(f"OPTIMIZE delta.`{gold_kpis_path}` ZORDER BY (city, vehicle_type)")

display(gold_kpis)

## 5. Delta Lake Governance: Audit History & Time Travel

In [ ]:
# Query Delta Lake Transaction Log History
delta_history = target_table.history()
display(delta_history.select("version", "timestamp", "operation", "operationMetrics"))

# Time Travel: Query Silver table as of Version 0 (Before Batch 2 Merge)
v0_df = spark.read.format("delta").option("versionAsOf", 0).load(silver_trips_path)
print(f"Version 0 Record Count: {v0_df.count()}")
print(f"Current Version Record Count: {silver_final.count()}")